# Practical 6: Simulating diffusion processes

Computational Finance with Python

[Alet Roux](https://www.york.ac.uk/maths/staff/alet-roux/) ([Department
of Mathematics](https://maths.york.ac.uk), University of York)

Click on the following to open this file in Google Colab:

<figure>
<a
href="https://colab.research.google.com/github/aletroux/comp-finance-python/blob/main/practicals/P06_simulating_diffusions_prac.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

The aim of this practical is to explore the simulation of diffusion
processes in Python.

# 1. Simulating Brownian motion

Let us first study how to simulate the trajectories of Brownian motion.
We are going to do this in two different ways.

First, let’s see how to do it using NumPy and loops. Study the following
code carefully to see how it works.

In [1]:
import numpy as np

import math

def simulate_BM_loops (rng, T, m, n):
    """Creates sample paths of Brownian motion using loops.
    Arguments:
    rng: random number generator
    T: time horizon
    m: number of steps
    n: number of paths
    Returns:
    W: Simulation. Each W[k] is a sample trajectory.
    """ 

    #time increment
    dt = T/m

    #increments of Brownian motion
    Z = rng.normal(0, math.sqrt(dt), (n, m))

    #reserve space for Brownian motion, insert initial value 0
    W = np.zeros((n, m+1))

    # The value W[k,0], the initial value of the k'th sample of 
    # Brownian motion, is 0 by the above.
    # For each i>0, the value W[k,i] is the sum of Z[k, :i].
    # in practice this means W[k,i] = W[k,i-1] + Z[k,i-1]
    for k in range(n):
        for i in range(m+1):
            W[k, i] = W[k, i-1] + Z[k, i-1]
    
    return W

# random number generator
rng = np.random.default_rng (seed = 30)

# use function simulate_BM_loops to simulate Brownian motion
T = 1
m = 100
n = 5
W = simulate_BM_loops (rng, T, m, n)

# now plot simulated paths
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (12,7))
t = np.linspace(0,T,m+1)
for k in range(n):
    ax.plot(t, W[k])

# plot cosmetics
ax.set(title = f"{n} simulated trajectories of Brownian motion with {m} steps")
ax.set(xlim = [0,T])
ax.xaxis.grid(True)
ax.yaxis.grid(True)

The following code simulates the trajectories of Brownian motion using
NumPy and vectorised code. Study it carefully to see how it works.

Notice the use of the `numpy.cumsum` function (NumPy Developers (2022)
provides more detail if needed). Can you spot any other differences?

In [2]:
def simulate_BM(rng, T, m, n):
    """Creates sample paths of Brownian motion.
    Arguments:
    rng: random number generator
    T: time horizon
    m: number of steps
    n: number of paths
    Returns:
    W: Simulation. Each W[k] is a sample trajectory.
    """ 

    #time increment
    dt = T/m

    #increments of Brownian motion
    Z = rng.normal(0, math.sqrt(dt), (n, m))

    #reserve space for Brownian motion, insert initial value 0
    W = np.zeros((n, m+1))

    # For each k, the value W[k,i] is the sum of Z[k, :i].
    # As the sum is taken over the columns, use the argument axis = 1
    # The alternative is to implement a Python loop, which is slow
    W[:, 1:] = np.cumsum(Z, axis = 1)
    
    return W

# random number generator
rng = np.random.default_rng (seed = 30)

# use function simulate_BM to simulate Brownian motion
T = 1
m = 100
n = 5
W = simulate_BM (rng, T, m, n)

# now plot simulated paths
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (12,7))
t = np.linspace(0,T,m+1)
for k in range(n):
    ax.plot(t, W[k])

# plot cosmetics
ax.set(title = f"{n} simulated trajectories of Brownian motion with {m} steps")
ax.set(xlim = [0,T])
ax.xaxis.grid(True)
ax.yaxis.grid(True)

The two code cells above produce the same output. However one of them
runs significantly faster than the other. Execute the following code
cells to compare the efficiency of the code. What do you notice?

In [3]:
%timeit simulate_BM_loops (rng, T, m, n)

In [4]:
%timeit simulate_BM (rng, T, m, n)

# 2. Simulating the Black-Scholes stock price

The Lamperti transform scheme for the Black-Scholes model reads as
follows:

1.  Set $\hat{Y}_0 = 0$.
2.  Set $\Delta t = T/m$.
3.  For $k=0,1,\ldots,n$:
    1.  For $i=0,1,\ldots,m-1$:
        1.  Generate $Z_{i+1}\sim N(0,1)$.
        2.  Set $\hat{Y}_{(i+1)\Delta t} = \hat{Y}_{i\Delta t} + \left(r - \tfrac{1}{2}\sigma^2\right)\Delta t +  \sigma\sqrt{\Delta t}Z_{i+1}$.
        3.  Set $\hat{S}_{k,(i+1)\Delta t} = S_0 e^{\hat{Y}_{(i+1)\Delta t}}$.

<span class="theorem-title">**Exercise 1**</span> Write Python code to
perform a simulation of the Black-Scholes model.

Use the code provided in the following cell to guide your work.

In [5]:
S0 = 80
r = 0.05
sigma = 0.15
T = 1

def simulate_BS(rng, S0, T, r, sigma, m, n):
    """Creates sample paths of Black-Scholes model.
    Arguments:
    rng: random number generator
    S0: initial value
    T: time horizon
    r: interest rate
    sigma: volatility
    m: number of steps
    n: number of paths
    Returns:
    S: Simulation. Each S[k] is a sample trajectory.
    """ 

    dt = T/m
    sqrdt = math.sqrt(dt)
    
    # Insert your code here.
    # You can use the function simulate_BM or simulate_BM_loops.
    # Or you can use its contents as a starting point for your work.

    # Modify the following line.
    S = np.tile(S0,(n,m+1))

    return S

# random number generator
rng = np.random.default_rng (seed = 30)

# use function simulate_BS to simulate Black-Scholes stock prices
T = 1
m = 100
n = 10
S = simulate_BS (rng, S0, T, r, sigma, m, n)

# now plot simulated paths
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (12,7))
t = np.linspace(0,T,m+1)
for k in range(n):
    ax.plot(t, S[k])

# plot cosmetics
ax.set(title = f"{n} simulated trajectories of Black-Scholes stock price with {m} steps")
ax.set(xlim = [0,T])
ax.plot(t, S0*np.exp((r-sigma**2/2)*t), linestyle = "dashed", 
        label = f"$y = {S0}e^{{ {round(r-sigma**2/2,5)}t}}$")
ax.legend()
ax.xaxis.grid(True)
ax.yaxis.grid(True)

You are ready to start work on Assignment 3.

# References

NumPy Developers. 2022. “Numpy.cumsum.”
<https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html>.